In [1]:
%pip install youtube-search

Note: you may need to restart the kernel to use updated packages.


In [10]:
from youtube_search import YoutubeSearch

videos=YoutubeSearch("미국 대선", max_results=5).to_dict()
videos

[{'id': '4A3P9Tgm8wc',
  'thumbnails': ['https://i.ytimg.com/vi/4A3P9Tgm8wc/hq720.jpg?sqp=-oaymwEjCOgCEMoBSFryq4qpAxUIARUAAAAAGAElAADIQj0AgKJDeAE=&rs=AOn4CLBiKa6FlGtJbPpVcHb4Nkfj_G7yqQ',
   'https://i.ytimg.com/vi/4A3P9Tgm8wc/hq720.jpg?sqp=-oaymwEXCNAFEJQDSFryq4qpAwkIARUAAIhCGAE=&rs=AOn4CLA9dp_amuxkebdEB0E_Yg87rC4ZQA'],
  'title': '[2020 미국 대선] 표 적게 받고도 대통령 된다? 복잡한 선거제도 6분 총정리 ❙ Won Less, but Elected Anyways? / 비디오머그',
  'long_desc': None,
  'channel': '비디오머그 - VIDEOMUG',
  'duration': '5:50',
  'views': '조회수 517,314회',
  'publish_time': '4년 전',
  'url_suffix': '/watch?v=4A3P9Tgm8wc&pp=ygUN66-46rWtIOuMgOyEoA%3D%3D'},
 {'id': 'rY5_-jEUJYw',
  'thumbnails': ['https://i.ytimg.com/vi/rY5_-jEUJYw/hq720.jpg?sqp=-oaymwEjCOgCEMoBSFryq4qpAxUIARUAAAAAGAElAADIQj0AgKJDeAE=&rs=AOn4CLBYYnU7eCb5PSpQCM3gDhOf1bt73g',
   'https://i.ytimg.com/vi/rY5_-jEUJYw/hq720.jpg?sqp=-oaymwEXCNAFEJQDSFryq4qpAwkIARUAAIhCGAE=&rs=AOn4CLDcZRS4dfbO_u3K-XKj5S99g25Wiw'],
  'title': '[LIVE] 2024 미국 대선 결과 분석 정리',
  'long_desc

In [11]:
video_url='https://youtube.com' + videos[3]['url_suffix']
video_url

'https://youtube.com/watch?v=Rx3dYZPHGAQ&pp=ygUN66-46rWtIOuMgOyEoA%3D%3D'

In [3]:
%pip install youtube_transcript_api

Note: you may need to restart the kernel to use updated packages.


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
print(hasattr(YouTubeTranscriptApi, "list_transcripts"))  # True여야 정상

False


In [6]:
%pip install --upgrade langchain langchain-community

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -U youtube-transcript-api

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -U pytube

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.documents import Document

# 구/신 API 자동 감지
try:
    from youtube_transcript_api import YouTubeTranscriptsApi as _YTNew
    _YTNew_OK = True
except ImportError:
    _YTNew = None
    _YTNew_OK = False

try:
    from youtube_transcript_api import YouTubeTranscriptApi as _YTOld
    _YTOld_OK = True
except ImportError:
    _YTOld = None
    _YTOld_OK = False

try:
    from youtube_transcript_api import NoTranscriptFound, TranscriptsDisabled
except ImportError:
    class NoTranscriptFound(Exception): ...
    class TranscriptsDisabled(Exception): ...

def _get_api():
    if _YTNew_OK:
        return _YTNew(), True   # (instance, is_new_api)
    if _YTOld_OK:
        return _YTOld, False    # (class, is_new_api)
    raise ImportError("youtube_transcript_api 가 설치되어 있지 않습니다.")

def _patched_load(self):
    # (옵션) 영상 메타데이터
    if getattr(self, "add_video_info", False):
        self._metadata.update(self._get_video_info())

    api, is_new = _get_api()

    # Transcript 목록 얻기
    try:
        if is_new:
            tlist = api.list(self.video_id)               # 신 API
        else:
            tlist = api.list_transcripts(self.video_id)   # 구 API
    except TranscriptsDisabled:
        return []

    # 언어 선택
    langs = self.language if isinstance(self.language, (list, tuple)) else [self.language]
    try:
        t = tlist.find_transcript(langs)
    except NoTranscriptFound:
        t = tlist.find_transcript(["en"])

    # 번역 옵션
    if getattr(self, "translation", None):
        t = t.translate(self.translation)

    fetched = t.fetch()

    # 결과 정규화: [{text,start,duration}, ...] 형태로 통일
    if isinstance(fetched, list):
        pieces = fetched  # 구 API: 이미 dict 리스트
    else:
        # 신 API 객체: to_raw_data() 또는 snippets 속성 사용
        if hasattr(fetched, "to_raw_data"):
            pieces = fetched.to_raw_data()
        elif hasattr(fetched, "snippets"):
            pieces = [{"text": s.text, "start": s.start, "duration": s.duration} for s in fetched.snippets]
        else:
            # 최후의 방어
            try:
                pieces = list(fetched)
            except Exception:
                pieces = [{"text": str(fetched), "start": 0.0, "duration": 0.0}]

    fmt = getattr(self, "transcript_format", None)
    fmt_val = getattr(fmt, "value", "text")

    if fmt_val == "text":
        text = " ".join(p.get("text","").strip() for p in pieces if p.get("text"))
        return [Document(page_content=text, metadata=self._metadata)]
    else:  # lines/chunks 등
        return [
            Document(
                page_content=p.get("text","").strip(),
                metadata={**self._metadata, "start": p.get("start"), "duration": p.get("duration")}
            )
            for p in pieces
        ]

# 실제 패치
YoutubeLoader.load = _patched_load


In [4]:
%pip install -U youtube-transcript-api yt-dlp webvtt-py

Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.documents import Document

# 구/신 youtube-transcript-api 자동 감지
try:
    from youtube_transcript_api import YouTubeTranscriptsApi as _YTNew
    _YTNew_OK = True
except ImportError:
    _YTNew = None
    _YTNew_OK = False

try:
    from youtube_transcript_api import YouTubeTranscriptApi as _YTOld
    _YTOld_OK = True
except ImportError:
    _YTOld = None
    _YTOld_OK = False

try:
    from youtube_transcript_api import NoTranscriptFound, TranscriptsDisabled
except ImportError:
    class NoTranscriptFound(Exception): ...
    class TranscriptsDisabled(Exception): ...

from xml.etree.ElementTree import ParseError as ETParseError
from xml.parsers.expat import ExpatError

def _fallback_with_ytdlp(url: str, langs):
    import os, glob, tempfile
    from yt_dlp import YoutubeDL
    import webvtt

    langs = list(langs)
    with tempfile.TemporaryDirectory() as td:
        opts = {
            "skip_download": True,
            "writesubtitles": True,
            "writeautomaticsub": True,
            "subtitlesformat": "vtt",
            "subtitleslangs": langs,           # 우선 언어
            "outtmpl": os.path.join(td, "%(id)s.%(ext)s"),
            "quiet": True,
            "no_warnings": True,
        }
        with YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)

        vid = info.get("id")
        # 언어 우선순위대로 파일 탐색
        candidates = []
        for lg in langs + ["ko", "en"]:
            candidates += glob.glob(os.path.join(td, f"{vid}.{lg}.vtt"))
        if not candidates:
            candidates = glob.glob(os.path.join(td, f"{vid}.*.vtt"))
        if not candidates:
            return []

        path = candidates[0]
        text = " ".join(c.text.strip() for c in webvtt.read(path))
        return [Document(page_content=text, metadata={"source": url})]

def _get_api(cookies_path: str | None = None):
    # cookies_path를 주면 신 API에 우선 적용
    if _YTNew_OK:
        kwargs = {}
        if cookies_path:
            kwargs["cookies"] = cookies_path  # 연령/지역 제한 대응
        return _YTNew(**kwargs), True
    if _YTOld_OK:
        return _YTOld, False
    raise ImportError("youtube_transcript_api 가 설치되어 있지 않습니다.")

def _patched_load(self):
    # (옵션) 영상 메타
    if getattr(self, "add_video_info", False):
        self._metadata.update(self._get_video_info())

    # 언어 우선순위 정리
    langs = self.language if isinstance(self.language, (list, tuple)) else [self.language]
    url = f"https://youtu.be/{self.video_id}"

    # 필요 시 쿠키 경로를 self.cookies에서 읽어 적용 (없어도 됨)
    cookies_path = getattr(self, "cookies", None)

    api, is_new = _get_api(cookies_path)

    # 1) 자막 목록
    try:
        tlist = api.list(self.video_id) if is_new else api.list_transcripts(self.video_id)
    except TranscriptsDisabled:
        # yt-dlp 백업 시도
        return _fallback_with_ytdlp(url, langs)

    # 2) 언어 선택
    try:
        t = tlist.find_transcript(langs)
    except NoTranscriptFound:
        try:
            t = tlist.find_transcript(["en"])
        except NoTranscriptFound:
            # yt-dlp 백업 시도
            return _fallback_with_ytdlp(url, langs)

    # 3) 번역 옵션
    if getattr(self, "translation", None):
        t = t.translate(self.translation)

    # 4) 실제 fetch + 예외 처리
    try:
        fetched = t.fetch()
    except (ETParseError, ExpatError) as e:
        # XML 파싱 실패 → yt-dlp로 백업
        return _fallback_with_ytdlp(url, langs)
    except Exception:
        # 그 외 예외도 백업 시도
        out = _fallback_with_ytdlp(url, langs)
        if out:
            return out
        raise

    # 5) 포맷 통일
    if isinstance(fetched, list):
        pieces = fetched  # 구 API: [{text,start,duration}, ...]
    elif hasattr(fetched, "to_raw_data"):
        pieces = fetched.to_raw_data()
    elif hasattr(fetched, "snippets"):
        pieces = [{"text": s.text, "start": s.start, "duration": s.duration} for s in fetched.snippets]
    else:
        pieces = [{"text": str(fetched), "start": 0.0, "duration": 0.0}]

    fmt = getattr(self, "transcript_format", None)
    fmt_val = getattr(fmt, "value", "text")

    if fmt_val == "text":
        text = " ".join(p.get("text","").strip() for p in pieces if p.get("text"))
        return [Document(page_content=text, metadata=self._metadata)]
    else:  # lines/chunks 등
        return [
            Document(
                page_content=p.get("text","").strip(),
                metadata={**self._metadata, "start": p.get("start"), "duration": p.get("duration")}
            )
            for p in pieces
        ]

# 메서드 교체
YoutubeLoader.load = _patched_load


In [3]:
from langchain_community.document_loaders import YoutubeLoader

loader=YoutubeLoader.from_youtube_url(
    video_url,
    language=['ko','en'] # 자막 언어
)

docs=loader.load()

AttributeError: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'

In [8]:
from langchain_core.documents import Document
from pytube import YouTube
from youtube_transcript_api import YouTubeTranscriptApi

# replace the 'YoutubeLoader' import

# Assuming 'video_url' is already defined.
video_id = YouTube(video_url).video_id

# Use the youtube-transcript-api to get the transcript
# Note: This is a different way of calling the same library, which might work.
try:
    transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['ko', 'en'])

    # Join the transcript parts into a single string
    full_text = " ".join([t['text'] for t in transcript])

    # Create a Document object
    docs = [Document(page_content=full_text, metadata={"source": video_url})]

    print("Transcript loaded successfully!")
    print(docs[0].page_content[:200]) # Print the first 200 characters to verify
except Exception as e:
    print(f"An error occurred: {e}")
    docs = [] # Set docs to an empty list on failure

An error occurred: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'


In [13]:
from langchain_core.documents import Document
from pytube import YouTube

# Assuming 'video_url' is already defined.
video_url='https://www.youtube.com/watch?v=Rx3dYZPHGAQ'
yt = YouTube(video_url)

# Attempt to get the transcript using pytube's internal methods
try:
    # Get the a list of available captions
    caption = yt.captions.get_by_language('ko')

    # If Korean captions are not available, try English
    if caption is None:
        caption = yt.captions.get_by_language('en')

    # Get the transcript
    if caption:
        transcript_xml = caption.xml_captions
        # You'll need to parse this XML to extract the text.
        # This is a bit more complex, so let's simplify for now.
        print("Pytube found captions. Proceed with manual XML parsing or use an online tool to test.")
        # For simplicity, we can get the text directly if not XML formatted
        full_text = caption.generate_srt_captions()
        
        # Create a Document object
        docs = [Document(page_content=full_text, metadata={"source": video_url})]

        print("Transcript loaded successfully!")
        print(docs[0].page_content[:200]) # Print the first 200 characters to verify
    else:
        print("No Korean or English captions found for this video.")
        docs = []

except Exception as e:
    print(f"An error occurred: {e}")
    docs = [] # Set docs to an empty list on failure

An error occurred: HTTP Error 400: Bad Request


In [5]:
%pip show langchain youtube-transcript-api

Name: langchain
Version: 0.3.27
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: d:\GPT_AGENT_2025_BOOK\venv\Lib\site-packages
Requires: langchain-core, langchain-text-splitters, langsmith, pydantic, PyYAML, requests, SQLAlchemy
Required-by: langchain-community, langchain-tavily
---
Name: youtube-transcript-api
Version: 1.2.2
Summary: This is an python API which allows you to get the transcripts/subtitles for a given YouTube video. It also works for automatically generated subtitles, supports translating subtitles and it does not require a headless browser, like other selenium based solutions do!
Home-page: https://github.com/jdepoix/youtube-transcript-api
Author: Jonas Depoix
Author-email: jonas.depoix@web.de
License: MIT
Location: d:\GPT_AGENT_2025_BOOK\venv\Lib\site-packages
Requires: defusedxml, requests
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install --upgrade youtube-transcript-api

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip list

Package                                  Version
---------------------------------------- ------------
accelerate                               1.10.1
aiohappyeyeballs                         2.6.1
aiohttp                                  3.12.15
aiosignal                                1.4.0
alembic                                  1.16.5
altair                                   5.5.0
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                                    4.10.0
asteroid-filterbanks                     0.4.0
asttokens                                3.0.0
attrs                                    25.3.0
audioread                                3.0.1
backoff                                  2.2.1
bcrypt                                   4.3.0
beautifulsoup4                           4.13.5
blinker                                  1.9.0
Brotli                                   1.1.0
build                                    1.3

In [17]:
%pip uninstall langchain_community

^C
Note: you may need to restart the kernel to use updated packages.


In [16]:
from langchain_community.document_loaders.youtube import YouTubeLoader
from langchain_core.documents import Document

# Replace with the URL you're trying to load
video_url = 'https://www.youtube.com/watch?v=Rx3dYZPHGAQ'

try:
    loader = YouTubeLoader.from_youtube_url(video_url, add_video_info=True)
    docs = loader.load()

    if docs:
        print("Transcript loaded successfully!")
        # Print the first 200 characters to verify
        print(docs[0].page_content[:200]) 
        # Print the metadata to confirm it's working
        print(docs[0].metadata)
    else:
        print("Failed to load transcript. Docs list is empty.")
except Exception as e:
    print(f"An error occurred: {e}")
    docs = []

ImportError: cannot import name 'YouTubeLoader' from 'langchain_community.document_loaders.youtube' (d:\GPT_AGENT_2025_BOOK\venv\Lib\site-packages\langchain_community\document_loaders\youtube.py)